# #4 Tracing performance

## Purpose

Evaluate tracing performance in the full dataset

## Setup

In [1]:
import treedata as td
import pycea as py
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from numba import njit
import matplotlib.ticker as mticker

from devmap.config import set_theme, stage_palette, lineage_palette, get_paths, embryos, type_palette
from devmap.config import stage_palette
from devmap.utils import save_plot, load_data
from devmap.topology import marked_branches_by_depth

set_theme()
base_path, plots_path, data_path = get_paths("validation")

## Load data

In [2]:
tdata = load_data('topology', characters = True)

## PE expression vs edit fraction

In [119]:
tdata.obs["pe_expr"] = np.log((tdata.obs["pe_counts"] / tdata.obs["total_counts"]) * 2e4 + 1)
df = tdata.obs.query("type == 'donor'").groupby(["cell_type","stage"]).agg(
    {"pe_counts": "mean", "edit_frac": "mean","pe_expr": "mean"}).reset_index()
fig, ax = plt.subplots(figsize=(2, 2), dpi=600, layout="constrained")
sns.scatterplot(data = df,x = "pe_expr",y = "edit_frac", hue = "stage", palette=stage_palette, ax = ax, s = 10)
plt.ylabel("Cell type mean edit fraction")
plt.xlabel("Cell type mean PE2maxGFP expression")
plt.xlim(0, 4)
plt.ylim(0, 0.7)
save_plot(plots_path / "pe_expr_vs_edit_frac.svg")

## Relative abundance

In [186]:
type_counts = tdata.obs.groupby(["cell_subtype","type","lineage"]).size()
type_counts = type_counts.unstack("type").fillna(0).reset_index()
type_counts["donor"] = type_counts["donor"] + 1
type_counts["host"] = type_counts["host"] + 1
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
plt.plot([0.6, 1e6], [0.6, 1e6], color="gray", linestyle="--", linewidth=1)
sns.scatterplot(data=type_counts, x="host", y="donor", hue="lineage", legend = False, s = 10, palette = lineage_palette)
plt.ylim(.6, 2e5)
plt.xlim(.6, 2e5)
plt.xscale("log")
plt.yscale("log")
for axis in [ax.xaxis, ax.yaxis]:
    axis.set_major_locator(mticker.LogLocator(base=10.0, numticks=100))
    axis.set_minor_locator(mticker.LogLocator(base=10.0, subs='auto', numticks=100))
plt.ylabel("Donor cell count")
plt.xlabel("Host cell count")
save_plot(plots_path / "donor_host_subtype_counts.svg", fig, transparent=True)

## Chimerism rate

In [11]:
tdata.obs["embryo"] = pd.Categorical(tdata.obs["embryo"], categories=reversed(embryos), ordered=True)
counts = (
    tdata.obs
    .groupby(['embryo', 'type'], observed=False)
    .size()
    .unstack(fill_value=0)
)
percent = counts.div(counts.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(1.2,2.5), dpi=600)
percent.plot(kind='barh', stacked=True, color = type_palette, ax = ax,edgecolor='black', linewidth=.5)
# remove legend
ax.legend().remove()
ax.set_xlabel("Percent")
ax.set_xticks([0, 50, 100])
plt.tight_layout()
save_plot(plots_path / "embryo_type_composition.svg", fig, transparent=True)

## Detection rate

In [225]:
fig, ax = plt.subplots(figsize=(1.5,2.5), dpi=600)
obs = tdata.obs.copy()
obs["detection_pct"] = obs["detection_rate"] * 100
sns.boxplot(obs.query("clone.notnull()").sample(100000), y = "embryo", x = "detection_pct", linecolor="black", linewidth=.6,
    order = embryos, showfliers = False, hue = "stage", palette=stage_palette, ax = ax, legend = False,saturation = 1,
    medianprops={"linewidth": 1})
# add star at mean for each embryo
means = obs.query("clone.notnull()").groupby("embryo")["detection_pct"].median()
plt.xlabel("Detection rate (%)") 
plt.ylabel("")
plt.xlim(60,105)
save_plot(plots_path / "embryo_detection_rate.svg", fig, transparent=True)

## Character vs tree distance

In [156]:
@njit
def norm_hamming_distance(arr1, arr2):
    valid_mask = (arr1 != -1) & (arr2 != -1)
    hamming_distance = 0
    for x, y in zip(arr1[valid_mask], arr2[valid_mask]):
        if x == y:
            pass
        elif x == 0 or y == 0:
            hamming_distance += 1
        else:
            hamming_distance += 2
    num_valid_comparisons = np.sum(valid_mask)
    if num_valid_comparisons == 0:
        return 0
    normalized_distance = hamming_distance / num_valid_comparisons
    return normalized_distance

py.tl.tree_distance(tdata,sample_n=20000, depth_key="time", update = False)
py.tl.distance(tdata, metric=norm_hamming_distance, key="characters", connect_key="tree_connectivities", update=False)

In [180]:
df = py.tl.compare_distance(tdata, dist_keys = ["tree","characters"]).query("obs1 != obs2")
df["tree_distances"] = df["tree_distances"] / 2
df["stage"] = df["obs1"].str.split("-").str[0]
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
sns.scatterplot(data=df.sample(frac=1), x="tree_distances", y="characters_distances", 
                alpha=0.5, hue = "stage", palette = stage_palette, s = 5, ax = ax)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Tree Distance (Days Since LCA)")
plt.ylabel("Normalized Hamming Distance")
save_plot(plots_path / "tree_vs_character_distance.svg", fig, transparent=True, rasterize=True)

## Number of extant cells over time

In [ ]:
n_extant = py.tl.n_extant(tdata,bins = np.arange(0,10.5,.5),copy = True, depth_key="time")
n_extant["embryo"] = n_extant["tree"].str.split("-C").str[0]
n_extant = n_extant.groupby(['time','embryo']).agg({'n_extant':'sum'}).reset_index()
n_extant["timepoint"] = n_extant["embryo"].str.split("-").str[0].str.replace("E", "").astype(float)
n_extant["stage"] = n_extant["embryo"].str.split("-R").str[0]

All embryos

In [ ]:
fig, ax = plt.subplots(figsize=(2,2),dpi=600, layout = "constrained")
sns.lineplot(data=n_extant.query("time <= timepoint"), x="time", y="n_extant", hue="stage", legend = False, 
             linewidth = 1, palette = stage_palette)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Embryonic time (days)")
plt.ylabel("Number of extant cells")
plt.yscale('log')
save_plot(plots_path / "growth_kinetics.svg", fig)

E9.5

In [ ]:
fig, ax = plt.subplots(figsize=(1.9,1.9),dpi=600, layout = "constrained")
sns.lineplot(data=n_extant.query("stage == 'E9.5' & time <= timepoint"), x="time", y="n_extant", hue="embryo", palette=embryo_palette, legend = False)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Embryonic time (days)")
plt.ylabel("Number of extant cells")
plt.yscale('log')
save_plot(plots_path / "e9.5_growth_kinetics.svg", fig)

## Fraction of branches marked by an edit

In [ ]:
marked_branches = []
for clone, tree in tdata.obst.items():
    df = marked_branches_by_depth(tree, bins=[0,2,4,6,7,8,9,10,11])
    marked_branches.append(df.assign(clone = clone))
marked_branches = pd.concat(marked_branches, ignore_index=True)

All embryos

In [ ]:
fig, ax = plt.subplots(figsize=(2,1.8),dpi=600, layout = "constrained")
marked_branches["embryo"] = marked_branches["clone"].str.split("-C").str[0]
marked_branches["stage"] = marked_branches["clone"].str.split("-").str[0]
sns.lineplot(data=marked_branches, x="bin_center", y="pct_marked", hue = "stage",palette=stage_palette,legend = False, linewidth = 1)
plt.ylabel("Resolved branches (%)")
plt.xticks([0,2,4,6,8,10])
plt.yticks([60,70,80,90,100])
plt.ylim(55,105)
plt.xlabel("Embryonic time (days)")
save_plot(plots_path / "marked_branches.svg", fig)

E9.5

In [ ]:
fig, ax = plt.subplots(figsize=(2,1.8),dpi=600, layout = "constrained")
marked_branches["embryo"] = marked_branches["clone"].str.split("-C").str[0]
sns.lineplot(data=marked_branches.query("stage == 'E9.5'"), x="bin_center", y="pct_marked", hue = "embryo",palette=embryo_palette,legend = False)
plt.ylabel("Resolved branches (%)")
plt.xticks([0,2,4,6,8,10])
plt.yticks([60,70,80,90,100])
plt.ylim(55,105)
plt.xlabel("Embryonic time (days)")
save_plot(plots_path / "e9.5_marked_branches.svg", fig)